In [1]:
import pandas as pd

In [2]:
matches = pd.read_csv(
    "../data/clean_matches.csv"
)

ranking = pd.read_csv(
    "../data/clean_rankings.csv"
)

team_stats = pd.read_csv(
    "../data/team_stats.csv"
)

In [3]:
matches["Date"] = pd.to_datetime(
    matches["Date"]
)

ranking["rank_date"] = pd.to_datetime(
    ranking["rank_date"]
)

In [4]:
matches = matches.sort_values(
    "Date"
)

ranking = ranking.sort_values(
    "rank_date"
)

In [5]:
TEAM_MAPPING = {
    "USA": "United States",
    "Korea Republic": "South Korea",
    "Korea DPR": "North Korea",
    "IR Iran": "Iran",
    "Türkiye": "Turkey"
}

matches["home_team"] = matches[
    "home_team"
].replace(TEAM_MAPPING)

matches["away_team"] = matches[
    "away_team"
].replace(TEAM_MAPPING)

In [6]:
home_ranking = ranking.rename(
    columns={
        "country_full": "home_team",
        "rank": "home_rank",
        "total_points": "home_points",
        "rank_date": "Date"
    }
)

In [7]:
matches = pd.merge_asof(
    matches,
    home_ranking,
    on="Date",
    by="home_team",
    direction="backward"
)

In [8]:
away_ranking = ranking.rename(
    columns={
        "country_full": "away_team",
        "rank": "away_rank",
        "total_points": "away_points",
        "rank_date": "Date"
    }
)

In [9]:
matches = pd.merge_asof(
    matches,
    away_ranking,
    on="Date",
    by="away_team",
    direction="backward"
)

In [10]:
matches["ranking_difference"] = (
    matches["away_rank"] -
    matches["home_rank"]
)

matches["points_difference"] = (
    matches["home_points"] -
    matches["away_points"]
)

In [11]:
home_stats = team_stats.rename(
    columns={
        "team": "home_team",
        "team_power_score": "home_team_power"
    }
)

In [12]:
matches = matches.merge(
    home_stats[
        [
            "home_team",
            "home_team_power"
        ]
    ],
    on="home_team",
    how="left"
)

In [13]:
away_stats = team_stats.rename(
    columns={
        "team": "away_team",
        "team_power_score": "away_team_power"
    }
)

In [14]:
matches = matches.merge(
    away_stats[
        [
            "away_team",
            "away_team_power"
        ]
    ],
    on="away_team",
    how="left"
)

In [15]:
matches["team_power_difference"] = (
    matches["home_team_power"] -
    matches["away_team_power"]
)

In [16]:
def get_result(row):

    if row["home_score"] > row["away_score"]:
        return "home_win"

    elif row["home_score"] < row["away_score"]:
        return "away_win"

    else:
        return "draw"


matches["result"] = matches.apply(
    get_result,
    axis=1
)

In [17]:
matches.to_csv(
    "../data/final_matches_dataset.csv",
    index=False
)

print("Dataset final salvo!")

Dataset final salvo!
